In [2]:
import requests
import json
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()
import time

In [4]:
entities = pd.read_csv("./data/metadata/all_entities.csv")

In [5]:
poliscope_api_key = os.getenv("POLISCOPE_API_KEY")
poliscope_api_url = os.getenv("POLISCOPE_API_URL")
# API headers
poliscope_headers = {
    "Authorization": f"Bearer {poliscope_api_key}",
    "x-client": "helen-integration"
}

# Search Terms definieren

Über die Keyword-Suche können wir alle Treffer zu bestimmten Suchbegriffen finden. Für die Erstellung der Suchbegriff-Liste bietet es sich an, zunächst über das Dashboard Inhalte zu explorieren, und/oder Clara zu fragen.



In [53]:
entity_ids = entities["id"].to_list()
search_terms_wp = ["Wärmeplanung", "Wärmeplan", "Fernwärme", "Fernwärmenetz", "Wärmenetz", "Wärmeversorgung", "Wärmeversorgungskonzept", "Wärmeversorgungskonzepte", "Wärmeplanungsgesetz"]
search_terms_autofrei = ["autofrei", "autofreie Stadt", "autofreies Stadtzentrum", "autofreie Innenstadt", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche"]
search_terms_autoarm = ["autoarm", "autoarme Stadt", "autoarmes Stadtzentrum", "autoarme Innenstadt", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche"]
search_terms = search_terms_autoarm
search_string = " OR ".join(search_terms)

In [52]:
len(set(entity_ids))

16077

In [16]:
search_string

'autofrei OR autofreie Stadt OR autofreies Stadtzentrum OR autofreie Innenstadt OR autofreie Innenstadtbereiche OR autofreie Innenstadtbereiche OR autofreie Innenstadtbereiche OR autofreie Innenstadtbereiche'

In [54]:
# Alle Items sammeln
all_items = []

# Paginierungsparameter
limit = 499
request_count = 0  # Zähler für Anfragen
max_requests_per_minute = 29  # Nach 29 Anfragen warten

print(f"\n🔍 Durchsuche {len(entity_ids)} Entity IDs...")

offset = 0
total_items = None

while True:
    response = requests.get(
        url=f"{poliscope_api_url}/search/scan",
        headers=poliscope_headers,
        params={
            "q": search_string,
            "limit": limit,
            "offset": offset
        }
    )

    request_count += 1

    if response.status_code != 200:
        print(f"  ✗ Error: {response.status_code}")
        break

    data = response.json()
    items = data.get("data", [])

    meta = data.get("meta", {})
    pagination = meta.get("pagination", {})
    total_items = pagination.get("total", 0)

    if not items:
        print("  → Keine weiteren Treffer.")
        break

    all_items.extend(items)
    offset += len(items)
    print(f"  → Downloaded {offset} / {total_items} items (Request #{request_count})")

    if offset >= total_items or len(items) < limit:
        break

    if request_count >= max_requests_per_minute:
        print("  ⏳ Rate limit erreicht. Warte 1 Minute...")
        time.sleep(60)
        request_count = 0

print(f"\n✓ FERTIG! Insgesamt {len(all_items)} items abgerufen.")


🔍 Durchsuche 16079 Entity IDs...
  → Downloaded 499 / 4331 items (Request #1)
  → Downloaded 998 / 4331 items (Request #2)
  → Downloaded 1497 / 4331 items (Request #3)
  → Downloaded 1996 / 4331 items (Request #4)
  → Downloaded 2495 / 4331 items (Request #5)
  → Downloaded 2994 / 4331 items (Request #6)
  → Downloaded 3493 / 4331 items (Request #7)
  → Downloaded 3992 / 4331 items (Request #8)
  → Downloaded 4331 / 4331 items (Request #9)

✓ FERTIG! Insgesamt 4331 items abgerufen.


In [55]:
all_items_df = pd.DataFrame(all_items)
all_items_df


,id,chunkType,groupKey,groupType,agendaItemId,documentId,entityId,entityLevel,date,proposalId
0,document:ac51b43f-fe14-4add-b2b5-e8deeb9cc8b1:692,document,meeting:bfd36b26-753a-4628-8a2e-0110f28920ca,meeting,96878ee4-5570-4fbe-baa4-26e6dada083a,ac51b43f-fe14-4add-b2b5-e8deeb9cc8b1,033520011,50,2022-12-01T16:35:00,NaN
1,document:d9de2705-9c29-45b7-91c9-c70e5f167f02:23,document,proposal:8aa0f96b-d6dc-4015-9971-c9cec4d13271,proposal,de339161-932f-4867-a1f7-7eb1147d4828,d9de2705-9c29-45b7-91c9-c70e5f167f02,051540064,50,2025-07-01T18:00:00,8aa0f96b-d6dc-4015-9971-c9cec4d13271
2,document:31b8f91e-42cf-4156-a729-ed39af14f1d3:437,document,meeting:a6f08ac4-1963-45aa-b457-86c67a391d0d,meeting,NaN,31b8f91e-42cf-4156-a729-ed39af14f1d3,057740032,50,2024-09-11T17:00:00,NaN
3,document:1b740879-7464-42ba-9a8e-969a72372db4:261,document,meeting:339afc1e-308e-49c9-bd33-e9c850888ba1,meeting,NaN,1b740879-7464-42ba-9a8e-969a72372db4,051540064,50,2024-12-10T17:00:00,NaN
4,document:a57b8b69-3d35-46ed-9cb5-901b1c1254ce:117,document,proposal:bdd56183-07d3-4152-ae9f-504f6e178f82,proposal,00dfc818-4377-4782-844f-1887f1c4e9af,a57b8b69-3d35-46ed-9cb5-901b1c1254ce,05111,40,2025-03-20T15:04:00,bdd56183-07d3-4152-ae9f-504f6e178f82
...,...,...,...,...,...,...,...,...,...,...
4326,document:1fd0915d-551f-46fd-ab78-92028518cf2c:9,document,proposal:7c9505d0-29a4-4b93-8f5c-265138ab7dfa,proposal,07f8b74a-15e1-4078-870f-fa9624de1702,1fd0915d-551f-46fd-ab78-92028518cf2c,160770001,50,2024-11-28T17:02:00,7c9505d0-29a4-4b93-8f5c-265138ab7dfa
4327,document:06990e95-1f3e-4746-9c81-3ad47b34692e:214,document,meeting:59f4fcc6-aa58-484e-8c29-5168e22e50dd,meeting,9c54778e-d9b1-4cc5-b207-22c44d326267,06990e95-1f3e-4746-9c81-3ad47b34692e,091790121,50,2025-12-16T20:14:00,NaN
4328,proposalDescription:ba4c7181-2df7-48f0-a7b9-2c...,proposalDescription,proposal:ba4c7181-2df7-48f0-a7b9-2c91b65e3625,proposal,7f1a652c-f6d4-42d9-bb6a-831e7803c200,NaN,05314,40,2026-07-15T18:00:00,ba4c7181-2df7-48f0-a7b9-2c91b65e3625
4329,document:a40ae073-4b42-4e45-a7ce-d97dc82310d0:13,document,meeting:ba640c43-e9a8-40f4-b12a-23d439b16bc7,meeting,NaN,a40ae073-4b42-4e45-a7ce-d97dc82310d0,055660048048,60,2025-06-03T18:00:00,NaN


In [56]:
all_items_df.to_csv("./data/raw/autoarm_items.csv", index=False)

highlights: Sind die Character Positions in dem text string

agendaItemID ist die Verknüpfung zu einer Sitzung

vmtl ist proposalID leer, wenns keins gibt